In [47]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict,Literal,Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
import operator
load_dotenv()

True

In [48]:
generator_llm = ChatGroq(model = "llama-3.3-70b-versatile")
evaluator_llm = ChatGroq(model = "qwen/qwen3-32b")
optimizer_llm = ChatGroq(model = "llama-3.3-70b-versatile")

In [49]:
from pydantic import BaseModel, Field
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved","need_improvement"] = Field(description="Give sentiment of feedback")
    feedback: str = Field(description="feedback of tweet")

In [50]:
structured_evaluator = evaluator_llm.with_structured_output(TweetEvaluation)

In [51]:
class TweetState(TypedDict):
    topic:str
    tweet: str
    feedback: str
    iteration: int
    max_iterations: int
    evaluation: Literal["approved","need_improvement"] 

    tweet_history: Annotated[list[str],operator.add]
    feedback_history: Annotated[list[str],operator.add]
    

In [52]:
def generate_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]
    tweet = generator_llm.invoke(messages).content
    return {"tweet":tweet,"tweet_history":[tweet]}

In [53]:
def evaluate_tweet(state:TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    response = structured_evaluator.invoke(messages)
    return {"evaluation":response.evaluation,"feedback":response.feedback,"feedback_history":[response.feedback]}

In [59]:
def optimize_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    print("Hello Ravindra!")

    response = optimizer_llm.invoke(messages).content
    iteration = state["iteration"] + 1

    return {'tweet': response,"iteration":iteration,"tweet_history":response}

In [55]:
def route_evalutaion(state: TweetState) -> Literal["approved","need_improvement"]:
    if state["evaluation"] == "approved" or state["iteration"] >= state["max_iterations"]:
        return "approved"
    else:
        return "needs_improvement"


In [56]:
graph = StateGraph(TweetState)

graph.add_node("generate_tweet",generate_tweet)
graph.add_node("evaluate_tweet",evaluate_tweet)
graph.add_node("optimize_tweet",optimize_tweet)

graph.add_edge(START,"generate_tweet")
graph.add_edge("generate_tweet","evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet",route_evalutaion,{"approved":END,"needs_improvement":"optimize_tweet"})
graph.add_edge("optimize_tweet","evaluate_tweet")
workflow = graph.compile()

In [60]:
inital_state = {
    "topic":"edebge",
    "iteration" : 1,
    "max_iterations": 3
}

workflow.invoke(inital_state)

{'topic': 'edebge',
 'tweet': 'just spent 10 mins trying to autocorrect "edebge" and i think my phone is trolling me',
 'feedback': "The tweet is original, leveraging a relatable tech frustration with a specific, unexpected detail ('edebge') that adds freshness. It’s humorous in a low-stakes, self-deprecating way and concise enough to stop scrolling. The viral potential is solid, as autocorrect fails are a universal pain point, and the phrasing feels authentic without relying on tired formats. No structural flaws—no setup-punchline framework, no generic punchline tacked on. Minor weakness: the humor is mild rather than gut-splitting, but the specificity of the example and the 'trolling' twist keep it engaging.",
 'iteration': 1,
 'max_iterations': 3,
 'evaluation': 'approved',
 'tweet_history': ['just spent 10 mins trying to autocorrect "edebge" and i think my phone is trolling me'],
 'feedback_history': ["The tweet is original, leveraging a relatable tech frustration with a specific, 